# 给文档片段补充读者可能使用的说法

用户问：“集成多个弱学习器来提升整体预测效果的方法是什么？”相关定义在同一份南瓜书 PDF 中。原文使用“基础的学习器”，简化版 BM25 的前 4 个片段却被“弱学习器”等字面相近内容带到了其他章节。

一种直接的改法是在建立索引前，为包含特定原文短语的片段补充读者可能使用的同义说法。这属于 Document Augmentation（文档增补）的一种简化形式。这里把用于检索的文字和交给回答模型的原文分开保存：补充说法只用于检索，回答时始终只使用 PDF 原文，不把人工补充的判断当成原文依据。


## 先看原来的结果

本节直接读取同一份 PDF 已建立的文本片段，不重新切分 PDF，也不重新计算向量。这里使用 BM25，是为了清楚显示补充文字如何影响词语匹配；前 4 条结果的资料量保持不变。

In [1]:
import json
import sys
from pathlib import Path

def find_course_root(start: Path) -> Path:
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到教程数据目录，请从本节所在目录运行。")

course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import (
    ChunkEvidence,
    build_bm25_chunk_search,
    load_query_catalog,
    load_default_collection,
    load_default_chunks,
)
from common.nontraining_utils import load_annotation

cases = {case["id"]: case for case in load_query_catalog()}
case = cases["ensemble_learning_definition"]
collection = load_default_collection()
chunks = load_default_chunks(collection)
original_search = build_bm25_chunk_search(chunks)
original_results = original_search(case["query"], top_k=4)

print("问题：", case["query"])
print("检索后再读取必要页标注。")
print("资料来源：同一份南瓜书 PDF；比较返回数量：前 4 条")
print("原始 BM25 返回页：", [item.pages[0] for item in original_results])


问题： 集成多个弱学习器来提升整体预测效果的方法是什么
需要查阅的页： [88]
资料来源：同一份南瓜书 PDF；比较返回数量：前 4 条
原始 BM25 返回页： [101, 104, 108, 101]


## 按原文短语补一句检索说明

下面用一条人工确认过的同义词规则演示。规则按片段自身是否包含“组合多个基础的学习器”这一原文短语触发，遍历所有片段，不读取问题集中的目标页或参考答案。每个片段同时保留检索文字和回答原文；规则只改变检索文字，不改原文。

实际项目可以由编辑手工维护常见同义词，也可以让模型先生成一些可能的问题，再由人抽查。无论用哪种方式，都应把补充文字与原文分开保存。


In [2]:
SYNONYM_RULES = {
    "组合多个基础的学习器": "集成多个学习器；基础学习器也常被称为弱学习器。",
}


def add_search_description(chunk: dict) -> dict:
    answer_text = chunk["text"]
    descriptions = [
        description
        for source_phrase, description in SYNONYM_RULES.items()
        if source_phrase in answer_text
    ]
    description = " ".join(descriptions)
    return {
        **chunk,
        "answer_text": answer_text,
        "search_description": description,
        "search_text": f"{description} {answer_text}".strip(),
    }

augmented_chunks = [add_search_description(chunk) for chunk in chunks]
augmented_by_id = {
    chunk["chunk_id"]: chunk for chunk in augmented_chunks
}
changed = [
    chunk for chunk in augmented_chunks if chunk["search_description"]
]
search_index_chunks = [
    {
        **chunk,
        "text": chunk["search_text"],
    }
    for chunk in augmented_chunks
]
augmented_search = build_bm25_chunk_search(search_index_chunks)

def to_answer_evidence(search_results):
    return [
        ChunkEvidence(
            item.chunk_id,
            item.pages,
            augmented_by_id[item.chunk_id]["answer_text"],
            item.score,
        )
        for item in search_results
    ]

print(f"共 {len(chunks)} 个片段，本例补充了 {len(changed)} 个片段。")
for chunk in changed:
    print(
        "页码：",
        chunk["pages"],
        "；检索文字新增说法：",
        chunk["search_description"],
        "；回答部分仍使用原文，字符数：",
        len(chunk["answer_text"]),
    )


共 987 个片段，本例补充了 1 个片段。
页码： [88] ；检索文字新增说法： 集成多个学习器；基础学习器也常被称为弱学习器。 ；回答部分仍使用原文，字符数： 256


In [3]:
from common.eval_utils import emit_tutorial_audit

search_results = augmented_search(case["query"], top_k=4)
annotation = load_annotation(case["id"])

def result_metrics(results, expected_pages):
    pages = [int(page) for item in results for page in item.pages]
    expected = set(int(page) for page in expected_pages)
    found = expected.intersection(pages)
    return {
        'pages': pages,
        'first_required_rank': next((index for index, item in enumerate(results, 1) if expected.intersection(item.pages)), None),
        'required_page_coverage': len(found) / len(expected) if expected else 0.0,
    }

def first_target_rank(results, expected_pages):
    expected = set(expected_pages)
    return next(
        (
            rank
            for rank, item in enumerate(results, 1)
            if expected.intersection(item.pages)
        ),
        None,
    )

before_rank = first_target_rank(original_results, annotation["expected_pages"])
after_rank = first_target_rank(search_results, annotation["expected_pages"])
changed_ids = {chunk["chunk_id"] for chunk in changed}
unrelated_ahead = [
    item
    for rank, item in enumerate(search_results, 1)
    if after_rank
    and rank < after_rank
    and item.chunk_id not in changed_ids
]
print("补充前页码：", [item.pages[0] for item in original_results])
print("补充后页码：", [item.pages[0] for item in search_results])
print(
    "目标定义的排名：",
    before_rank or "前 4 个没有找到",
    "→",
    after_rank or "前 4 个没有找到",
)
print("目标前的无关片段数：", len(unrelated_ahead))
assert before_rank is None and after_rank == 1
assert not unrelated_ahead
print(
    "结论：补充检索说法把目标定义从前 4 条之外提到第 1 条，"
    "且目标前没有无关片段。"
)
emit_tutorial_audit({
    'case_id': 'ensemble_learning_definition',
    'method': '给片段补充检索说法',
    'role': 'main',
    'before': result_metrics(original_results, annotation['expected_pages']),
    'after': result_metrics(search_results, annotation['expected_pages']),
})


补充前页码： [101, 104, 108, 101]
补充后页码： [88, 101, 104, 108]
目标定义的排名： 前 4 个没有找到 → 1
目标前的无关片段数： 0
结论：补充检索说法把目标定义从前 4 条之外提到第 1 条，且目标前没有无关片段。


## 确认检索字段和回答字段分离

检索时可以同时使用补充说明和原文。交给回答模型时，再把命中的片段换回原文；这样生成的回答不会把人工补充说法误当成书中原话。


In [4]:
from common.eval_utils import emit_tutorial_audit

answer_results = to_answer_evidence(search_results)
top_result = search_results[0]
top_source = augmented_by_id[top_result.chunk_id]
top_answer = answer_results[0]

print(
    "用于检索的文字包含人工补充说法：",
    top_source["search_description"] in top_source["search_text"],
)
print(
    "交给回答的原文包含人工补充说法：",
    top_source["search_description"] in top_answer.text,
)
print(
    "交给回答的资料是否与原始片段完全一致：",
    top_answer.text == top_source["answer_text"],
)
print("回答字段片段：", top_answer.text[:360])
assert top_answer.text == top_source["answer_text"]
assert top_source["search_description"] not in top_answer.text

check_case = cases["sample_space_feature_engineering"]
check_original_results = original_search(check_case["query"], top_k=4)
check_augmented_results = augmented_search(check_case["query"], top_k=4)
check_annotation = load_annotation(check_case["id"])

def check_answer_text(results):
    return "".join(
        augmented_by_id[item.chunk_id]["answer_text"] for item in results
    )

def check_answer_points(results):
    text = check_answer_text(results).replace(" ", "")
    return {
        "样本空间的两个别称": all(
            term in text for term in ("样本空间", "输入空间", "属性空间")
        ),
    }

check_original_answer = to_answer_evidence(check_original_results)
check_augmented_answer = to_answer_evidence(check_augmented_results)
check_original_pages = [item.pages[0] for item in check_original_results]
check_augmented_pages = [item.pages[0] for item in check_augmented_results]
check_original_rank = first_target_rank(check_original_results, check_annotation["expected_pages"])
check_augmented_rank = first_target_rank(check_augmented_results, check_annotation["expected_pages"])
check_original_points = check_answer_points(check_original_results)
check_augmented_points = check_answer_points(check_augmented_results)
check_original_chars = sum(len(item.text) for item in check_original_answer)
check_augmented_chars = sum(len(item.text) for item in check_augmented_answer)

print("复核问题：", check_case["query"])
print("原始检索前 4 页：", check_original_pages)
print("补充后检索前 4 页：", check_augmented_pages)
print("目标页排名（原始 → 补充后）：", check_original_rank, "→", check_augmented_rank)
print("回答要点（原始 → 补充后）：", check_original_points, "→", check_augmented_points)
print("回答上下文字符数（原始 → 补充后）：", check_original_chars, "→", check_augmented_chars)
print("是否有补充片段被错误提前：", any(item.chunk_id in changed_ids for item in check_augmented_results))

assert check_original_pages == check_augmented_pages
assert check_original_rank == check_augmented_rank
assert check_original_points == check_augmented_points == {"样本空间的两个别称": True}
assert check_original_chars == check_augmented_chars
assert not any(item.chunk_id in changed_ids for item in check_augmented_results)
print("结论：正常查询的前 4 页、目标排名、回答要点和回答字符数均未变差。")
emit_tutorial_audit({
    'case_id': 'sample_space_feature_engineering',
    'method': '给片段补充检索说法',
    'role': 'check',
    'before': result_metrics(check_original_results, check_annotation['expected_pages']),
    'after': result_metrics(check_augmented_results, check_annotation['expected_pages']),
    'check_purpose': '确认没有改坏',
})


用于检索的文字包含人工补充说法： True
交给回答的原文包含人工补充说法： False
交给回答的资料是否与原始片段完全一致： True
回答字段片段： 第8章集成学习集成学习(ensemblelearning)描述的是组合多个基础的学习器（模型）的结果已达到更加鲁棒、效果更好的学习器。在“西瓜书”作者周志华教授的谷歌学术主页的top引用文章中，很大一部分都和集成学习有关。图8-3周志华教授谷歌学术top10引用文章(截止到2023-02-19)如图8-3所示。在引用次数前10的文章中，第1名“Top10algorithmsindatamining”是在ICDM’06中投票选出的数据挖掘十大算法，每个提名算法均由业内专家代表去阐述，然后进行投票，其中最终得票
复核问题： 样本空间的别称是什么？
原始检索前 4 页： [151, 14, 178, 133]
补充后检索前 4 页： [151, 14, 178, 133]
目标页排名（原始 → 补充后）： 2 → 2
回答要点（原始 → 补充后）： {'样本空间的两个别称': True} → {'样本空间的两个别称': True}
回答上下文字符数（原始 → 补充后）： 1024 → 1024
是否有补充片段被错误提前： False
结论：正常查询的前 4 页、目标排名、回答要点和回答字符数均未变差。


## 什么时候不适用

原来前 4 条结果中没有目标定义，按原文短语触发的补充说明后目标定义排到第 1 条，解决了读者说“弱学习器”而定义页写“基础的学习器”造成的没找到问题；同一查询的返回数量仍为 4 条。保存的输出还直接显示：人工说法只用于检索，没有进入交给回答模型的原文。对“样本空间”正常查询同时跑原始和补充后的检索，前 4 页、目标排名、回答要点和回答字符数都相同，证明补充说法没有把正常问题变差。

使用前要检查两件事：补充文字是否把无关片段带到前面，生成的问题是否包含原文没有的信息。批量生成这些说明时，保存生成规则和抽查结果即可；回答阶段始终只使用原文字段。


## Document Augmentation：让片段覆盖读者的问法

Document Augmentation（给片段补充检索说法）的做法是：先把 PDF 分成片段，再由语言模型为整段文档或单个片段生成它能够回答的问题，并把这些问题与原片段一起加入向量库。用户的问法更接近这些补充问题时，检索仍返回原片段。按整段文档生成问题，覆盖范围较广、调用次数较少；按片段生成更准确，但需要更多调用和存储。

一个完整流程是：

1. 清理 PDF 并按 token 预算得到文档/片段；
2. 用结构化输出要求模型只返回可由原文回答的问题；
3. 人工抽查问题是否真的由片段支持，去重并过滤过长或带答案的问题；
4. 把 search_text（原文 + 问题/同义说法）与 answer_text（原文）分开保存；
5. 检索命中后，按稳定 ID 取回原文，不能把模型生成的问题当成证据。

当前案例采用一条人工核对的同义词规则，专门处理“弱学习器”和原文“基础的学习器”的说法差异。这是 Document Augmentation 的轻量变体，不重新生成整份问题集；代码输出保留了正常查询的复查，避免补充词把无关片段提前。


## 生成检索说法的代码要点

本页正式实验使用人工核对的检索说法；下面只说明生成结果如何清洗、写入检索字段并保留原文，不产生另一份实验结果。

```python
def clean_generated_questions(questions, max_chars=120):
    cleaned = []
    for question in questions:
        question = question.strip().lstrip("0123456789. ")
        if question.endswith(("？", "?")) and len(question) <= max_chars:
            cleaned.append(question)
    return list(dict.fromkeys(cleaned))

def make_augmented_record(chunk, questions):
    original = chunk["text"]
    generated = "；".join(clean_generated_questions(questions))
    return {
        **chunk,
        "search_text": f"{generated}\n{original}".strip(),
        "answer_text": original,
        "augmentation": generated,
    }

# 生成问题时应把完整文档/当前片段传给模型，并记录模型版本。
# 生成后先人工抽查，再把 search_text 建索引；回答只使用 answer_text。
```
